# L05 · 경험으로 배우기: MC·TD·Q-learning

## Goal

- MC와 TD target을 비교한다
- off-policy target을 계산한다
- termination과 truncation을 나눈다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L05:toy:42").hexdigest()
print(f"lesson=L05 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L05 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:c7ad33c1f4e04332bf1b1303825d926c1144f26e9eddc8977bf61b78dda851de data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: MDP·Bellman → **MC·TD·Q-learning** → DQN

$$G_t=R_{t+1}+\gamma G_{t+1},\qquad y_t^{TD}=R_{t+1}+\gamma(1-d_t)V(S_{t+1})$$

MC는 episode 끝까지 관찰한 return을 쓰므로 편향은 작지만 분산이 큽니다. TD는 다음 value로 bootstrap해 더 일찍 배우지만 추정 오차를 물려받습니다. Q-learning target은 실제 다음 action이 아니라 최대 Q action을 써서 off-policy입니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** terminal transition의 next value가 99여도 TD target에 들어갈까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>들어가지 않습니다. `(1-d_t)`가 bootstrap을 0으로 만들고 target은 마지막 reward 1입니다.</details>

In [2]:
from rl_study.algorithms.tabular import (
    monte_carlo_returns, q_learning_target, td_target
)
rewards = torch.tensor([-0.01, -0.01, 1.0])
mc_targets = monte_carlo_returns(rewards, gamma=0.9)
td_targets = td_target(
    rewards, torch.tensor([0.7, 0.8, 99.0]),
    torch.tensor([False, False, True]), gamma=0.9
)
q_target = q_learning_target(
    torch.tensor([1.0]), torch.tensor([[2.0, 4.0]]),
    torch.tensor([False]), gamma=0.9
)
print({"mc": mc_targets.tolist(), "td": td_targets.tolist(),
       "off_policy_q_target": float(q_target[0])})

{'mc': [0.7909999489784241, 0.8899999856948853, 1.0], 'td': [0.6200000047683716, 0.7099999785423279, 1.0], 'off_policy_q_target': 4.599999904632568}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** 세 target을 같은 trajectory에 놓으면 variance, bootstrap, off-policy라는 차이 하나씩만 바뀝니다. 별도 학습 curve보다 먼저 analytic 값으로 구현 경계를 검산하기 좋습니다.

**흔한 함정:** 시간 제한 `truncated`를 환경 종결 `terminated`처럼 다루면 bootstrap 가능한 정보를 버립니다. 두 flag를 합치지 않는 API와 test가 필요합니다. 회귀 test: `test_q_terminal_target`, `test_q_off_policy_target`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert td_targets[-1].item() == 1.0
assert torch.allclose(q_target, torch.tensor([4.6]))
print("checks=passed")

checks=passed


**회상 문제:** MC target과 TD target 중 어느 쪽이 현재 value estimate에 직접 의존하나요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 출력에서 terminal TD target은 1.0으로 고정되고, Q-learning은 최대 next-Q 4를 사용해 4.6을 만들었습니다.
- 실제 확인: `test_q_terminal_target`, `test_q_off_policy_target`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L06에서 Q table을 neural network로 바꾸고 target network와 replay가 왜 필요한지 봅니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

[상세 구현 문서](../../docs/algorithms/classic.md) · [강좌 지도](../../docs/course-map.md)

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`